# Chatbot Intent Classification with Hugging Face Transformers

This notebook demonstrates a complete NLP pipeline for intent classification using a fine-tuned BERT model, including dataset loading, preprocessing, baseline modeling, transformer fine-tuning, evaluation, and a simple deployment interface with Gradio.

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate scikit-learn evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


## 1. Problem Definition

The objective is to classify user utterances into predefined intents. This is a multi-class text classification problem. The evaluation will primarily focus on accuracy and F1-score to assess the model's performance across all intent classes.

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import evaluate

## 2. Dataset Collection and Initial Exploration

We will use the 'snips_built_in_intents' dataset from the Hugging Face `datasets` library. This dataset provides user utterances and their corresponding intent labels.

In [ ]:
# Loading the SNIPS dataset (you can swap 'snips_built_in_intents' for your preferred dataset)
dataset = load_dataset("snips_built_in_intents")

print("Dataset Dictionary Structure:")
print(dataset)

# Inspect a single example
print("\nSample Training Example:")
print(dataset['train'][0])

# Extract unique intents to understand our classification space
num_labels = len(dataset['train'].features['label'].names)
label_names = dataset['train'].features['label'].names
print(f"\nNumber of Intent Classes: {num_labels}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/328 [00:00<?, ? examples/s]

Dataset Dictionary Structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 328
    })
})

Sample Training Example:
{'text': "Share my location with Hillary's sister", 'label': 5}

Number of Intent Classes: 10


## 3. Preprocessing

This section covers tokenization using a pre-trained tokenizer from the Hugging Face `transformers` library and splitting the dataset into training and validation sets.

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Padding and truncation ensure all sequences are the same length
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# Apply tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Format the dataset to return PyTorch tensors
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/328 [00:00<?, ? examples/s]

### Splitting the Dataset
To get a more reliable evaluation, we'll split the tokenized training data into a training set and a validation set.

In [ ]:
train_test_split_dataset = tokenized_datasets['train'].train_test_split(test_size=0.2)

# Rename the splits for clarity
train_dataset = train_test_split_dataset['train']
eval_dataset = train_test_split_dataset['test']

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(eval_dataset)}")

Training dataset size: 262
Validation dataset size: 66


## 4. Baseline Model (TF-IDF + Naive Bayes)

Before fine-tuning a transformer model, we establish a baseline using a simpler, traditional NLP approach: TF-IDF vectorization combined with a Multinomial Naive Bayes classifier. This helps to understand the performance improvement offered by more complex models.

In [ ]:
# Extract raw text and labels for the baseline
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']
test_texts = dataset['train']['text'][:500] # Using a subset for quick testing
test_labels = dataset['train']['label'][:500]

# Vectorize the text
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

# Train a Naive Bayes classifier
nb_model = MultinomialNB()
nb_model.fit(X_train, train_labels)

# Evaluate baseline
nb_predictions = nb_model.predict(X_test)
print("Baseline Naive Bayes Accuracy:", accuracy_score(test_labels, nb_predictions))

Baseline Naive Bayes Accuracy: 0.9237804878048781


## 5. Transformer Fine-tuning (BERT)

Here, we fine-tune a pre-trained BERT model for sequence classification on our intent classification task. We use the Hugging Face `Trainer` API for efficient training and evaluation.

In [ ]:
# Load pre-trained BERT with the correct number of labels
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Define training parameters
training_args = TrainingArguments(
    output_dir="./intent_results",
    eval_strategy="epoch",  # <--- This is the fixed line!
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Metric function for the Trainer
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["train"],
    compute_metrics=compute_metrics,
)

# Start fine-tuning
trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.888803,0.515244
2,No log,1.620450,0.597561
3,No log,1.510913,0.685976


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=63, training_loss=1.8538587055509053, metrics={'train_runtime': 941.7977, 'train_samples_per_second': 1.045, 'train_steps_per_second': 0.067, 'total_flos': 32364984379392.0, 'train_loss': 1.8538587055509053, 'epoch': 3.0})

## 6. Evaluation

We evaluate the fine-tuned BERT model using a detailed classification report, which provides precision, recall, and F1-score for each intent class, as well as overall accuracy.

In [ ]:
# Get predictions on the evaluation set
predictions_output = trainer.predict(tokenized_datasets["train"]) # Use test set in practice
predictions = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

# Print a detailed classification report (Precision, Recall, F1-Score)
print(classification_report(true_labels, predictions, target_names=label_names))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


                       precision    recall  f1-score   support

        ComparePlaces       0.77      0.89      0.83        19
          RequestRide       0.00      0.00      0.00        26
           GetWeather       0.85      0.93      0.89        42
          SearchPlace       0.62      0.36      0.45        28
      GetPlaceDetails       0.67      0.92      0.77        50
 ShareCurrentLocation       0.00      0.00      0.00        16
GetTrafficInformation       1.00      0.65      0.79        20
       BookRestaurant       0.54      0.99      0.70        70
        GetDirections       0.91      0.86      0.88        35
             ShareETA       1.00      0.05      0.09        22

             accuracy                           0.69       328
            macro avg       0.64      0.56      0.54       328
         weighted avg       0.65      0.69      0.61       328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 7. Inference and Deployment (Optional with Gradio)

This section demonstrates how to use the fine-tuned model for inference on new, unseen text inputs. We then create a simple web interface using Gradio to allow interactive testing of the intent classifier.

In [ ]:
from transformers import pipeline

# Create a classification pipeline using our fine-tuned model and tokenizer
intent_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Test some custom sentences
test_sentences = [
    "Add this track to my workout playlist.",
    "What is the weather going to be like tomorrow in Tokyo?",
    "Book a table for two at a Thai restaurant."
]

for sentence in test_sentences:
    result = intent_classifier(sentence)[0]
    # Map the predicted label ID back to the readable intent name
    label_id = int(result['label'].split('_')[-1]) if 'LABEL' in result['label'] else int(result['label'])
    intent_name = label_names[label_id] if 'LABEL' in result['label'] else result['label']

    print(f"User: '{sentence}'")
    print(f"Predicted Intent: {intent_name} (Confidence: {result['score']:.4f})\n")

User: 'Add this track to my workout playlist.'
Predicted Intent: BookRestaurant (Confidence: 0.2469)

User: 'What is the weather going to be like tomorrow in Tokyo?'
Predicted Intent: GetWeather (Confidence: 0.3031)

User: 'Book a table for two at a Thai restaurant.'
Predicted Intent: BookRestaurant (Confidence: 0.3986)



In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
from transformers import pipeline
import torch

# 1. Force the pipeline to rebuild on the correct hardware (GPU if available)
device = 0 if torch.cuda.is_available() else -1
intent_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

# 2. Define the prediction function with a safety net
def predict_intent(user_text):
    try:
        # Ignore empty submissions
        if not user_text.strip():
            return "Please type a message first."

        # Get prediction
        result = intent_classifier(user_text)[0]

        # Safely map the label back to the readable name
        if 'LABEL' in result['label']:
            label_id = int(result['label'].split('_')[-1])
            intent_name = label_names[label_id]
        else:
            try:
                label_id = int(result['label'])
                intent_name = label_names[label_id]
            except ValueError:
                intent_name = result['label'] # If it's already a text string

        return f"🎯 {intent_name}\n📊 Confidence: {result['score']:.2%}"

    except Exception as e:
        # If it breaks, tell us EXACTLY why on the screen
        return f"⚠️ Error: {str(e)}"

# 3. Build and launch the UI with debug mode ON
demo = gr.Interface(
    fn=predict_intent,
    inputs=gr.Textbox(lines=2, placeholder="Type a command here... e.g., 'Book a flight to Paris'"),
    outputs=gr.Textbox(label="Model Prediction"),
    title="🤖 Chatbot Intent Detection Prototype",
    height=500, # Increased height
    width=800   # Increased width
)

# debug=True forces Colab to print the exact error message in the cell output if it crashes again!
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9ee9e4a879d03771bb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Configure Git with your information
!git config --global user.name "najlaajou"
!git config --global user.email "najlaajou23@gmail.com"

# Clone your repository into the Colab environment
!git clone https://github.com/najlaajou/NLP-project.git

Cloning into 'NLP-project'...


In [ ]:
import os

# 1. Go back to the default Colab starting folder
os.chdir('/content')

# 2. Try to clone the repository again
print("Attempting to clone repository...")
!git clone https://github.com/najlaajou/NLP-project.git

# 3. Safety check: Did it actually create the folder?
if not os.path.exists('/content/NLP-project'):
    print("\n🚨 ERROR: The folder still wasn't created! 🚨")
    print("This means either:")
    print("  A) The repository doesn't exist on GitHub yet.")
    print("  B) The repository is Private and blocking Colab from seeing it.")
else:
    print("\n✅ Folder found! Creating project files...")

    # Move into the repository folder
    os.chdir('/content/NLP-project')

    # Create .gitignore
    with open('.gitignore', 'w') as f:
        f.write("intent_results/\nintent_results_v2/\n__pycache__/\n*.safetensors\n*.bin\n")

    # Create requirements.txt
    with open('requirements.txt', 'w') as f:
        f.write("transformers\ndatasets\ntorch\ngradio\nscikit-learn\nevaluate\nseaborn\nmatplotlib\n")

    # Create README.md
    readme_content = """# NLP Project 9: Chatbot Intent Detection
This repository contains a fine-tuned BERT model for intent detection using the SNIPS dataset.

## How to Run
1. Install requirements: `pip install -r requirements.txt`
2. Open the Colab notebook to train the model and launch the Gradio UI.
"""
    with open('README.md', 'w') as f:
        f.write(readme_content)

    print("✅ Files successfully created inside the NLP-project folder! You can proceed to Step 4.")

Attempting to clone repository...
Cloning into 'NLP-project'...

✅ Folder found! Creating project files...
✅ Files successfully created inside the NLP-project folder! You can proceed to Step 4.
